# Spotify Preprocess CLI-style Notebook

Streamed, two-pass preprocessing that mirrors the Python CLI script. Run top to bottom to produce a single consolidated `spotify_clean.csv` with lightweight memory use. Logs emulate a concise CLI output.

## How to run
1. Edit `CHUNK_SIZE`, `RAW_CSV`, or `OUT_CSV` in the run cell if needed.
2. Run all code cells below (they are self-contained).
3. Watch the structured logs for progress; the final message reports the saved file and row count.

In [1]:
# Imports and path detection
import argparse
import hashlib
import re
from pathlib import Path
import pandas as pd

def detect_app_root(start: Path) -> Path:
    base = start
    if base.name == '.venv':
        base = base.parent
    if (base / 'approot').exists():
        return (base / 'approot').resolve()
    p = base
    for _ in range(6):
        if (p / 'approot').exists():
            return (p / 'approot').resolve()
        if p.parent == p:
            break
        p = p.parent
    return base.resolve()

APP_ROOT = detect_app_root(Path.cwd())
RAW_CSV = APP_ROOT / 'data' / 'raw' / 'songs_with_attributes_and_lyrics.csv'
OUT_CSV = APP_ROOT / 'data' / 'processed' / 'spotify_clean.csv'
CHUNK_SIZE = 50_000  # adjust downward if memory is tight
print(f'app root      : {APP_ROOT}')
print(f'raw csv exists: {RAW_CSV.exists()}')
print(f'output target : {OUT_CSV}')

app root      : C:\Users\Aman Tewari\Documents\aaa-college\sem6\SLA-pvt\approot
raw csv exists: True
output target : C:\Users\Aman Tewari\Documents\aaa-college\sem6\SLA-pvt\approot\data\processed\spotify_clean.csv


In [ ]:
# Helpers
def log(stage: str, message: str) -> None:
    print(f'[{stage:>6}] {message}')


def ascii_ratio(text: str) -> float:
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return sum(ord(c) < 128 for c in text) / len(text)


def normalize_meta(s: str) -> str:
    if not isinstance(s, str):
        return ''
    s = s.lower()
    s = re.sub(r"\([^)]*(live|remaster(ed)?).*?\)", '', s, flags=re.IGNORECASE)
    s = re.sub(r"[\-\u2013\u2014]\s*live\s*$", '', s, flags=re.IGNORECASE)
    s = re.sub(r"\([^)]*\)", '', s)
    s = re.sub(r"[^\w\s]", ' ', s)
    return re.sub(r"\s+", ' ', s).strip()


def normalize_title_strong(s: str) -> str:
    if not isinstance(s, str):
        return ''
    t = s.lower()
    t = re.sub(r'^[^A-Za-z0-9]+|[^A-Za-z0-9]+$', '', t)
    t = re.sub(r'[^A-Za-z0-9 ]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def normalize_title(s: str) -> str:
    if not isinstance(s, str):
        return ''
    t = s.lower()
    t = re.sub(r'^[^A-Za-z0-9]+|[^A-Za-z0-9]+$', '', t)
    t = re.sub(r'[^A-Za-z0-9 ]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def normalize_lyrics(text: str) -> str:
    if not isinstance(text, str):
        return ''
    t = text.lower()
    t = re.sub(r"[^\w\s']", ' ', t)
    lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in t.splitlines()]
    return '\n'.join(lines)


def md5_hash(s: str) -> str:
    return hashlib.md5(s.encode('utf-8')).hexdigest()

In [ ]:
# Cleaning and streaming passes
RAW_COLS = ['id', 'name', 'artists', 'lyrics']


def clean_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk = chunk.rename(columns={'name': 'title', 'artists': 'artist'})
    chunk = chunk[chunk['lyrics'].notna()].copy()

    chunk['_word_count'] = chunk['lyrics'].astype(str).str.split().str.len()
    chunk = chunk[chunk['_word_count'] >= 30].copy()
    chunk['_ascii'] = chunk['lyrics'].astype(str).map(ascii_ratio)
    chunk = chunk[chunk['_ascii'] >= 0.90].copy()

    chunk['artist'] = chunk['artist'].astype(str).str.lower()
    chunk['artist'] = chunk['artist'].str.replace(r'\[|\]', '', regex=True)
    chunk['artist'] = chunk['artist'].str.replace("'", '', regex=False)
    chunk['artist'] = chunk['artist'].str.replace(r'\s+', ' ', regex=True).str.strip()

    chunk['title'] = chunk['title'].astype(str)
    chunk['normalized_title'] = chunk['title'].map(normalize_title_strong)

    chunk['normalized_artist'] = chunk['artist'].str.lower()
    chunk['normalized_artist'] = chunk['normalized_artist'].str.replace(r'[^A-Za-z0-9 ]+', ' ', regex=True)
    chunk['normalized_artist'] = chunk['normalized_artist'].str.replace(r'\s+', ' ', regex=True).str.strip()

    chunk['lyrics'] = chunk['lyrics'].astype(str).map(normalize_lyrics)

    chunk['normalized_lyrics'] = chunk['lyrics'].str.lower()
    chunk['normalized_lyrics'] = chunk['normalized_lyrics'].str.replace(r'[^\w\s]', ' ', regex=True)
    chunk['normalized_lyrics'] = chunk['normalized_lyrics'].str.replace(r'\s+', ' ', regex=True).str.strip()

    chunk['_comp_key'] = chunk['normalized_title'].fillna('') + '_' + chunk['normalized_artist'].fillna('')
    chunk['_lyrics_hash'] = chunk['lyrics'].map(md5_hash)
    return chunk


def pass1_compute_bounds(raw_csv: Path, chunk_size: int, usecols) -> tuple[int, int, int, int]:
    seen_comp_keys: set[str] = set()
    seen_lyrics_hash: set[str] = set()
    word_counts: list[int] = []
    rows_in = rows_kept = 0

    for i, chunk in enumerate(pd.read_csv(raw_csv, usecols=usecols, chunksize=chunk_size, low_memory=False)):
        rows_in += len(chunk)
        chunk = clean_chunk(chunk)
        mask_new = (~chunk['_comp_key'].isin(seen_comp_keys)) & (~chunk['_lyrics_hash'].isin(seen_lyrics_hash))
        chunk = chunk[mask_new]
        seen_comp_keys.update(chunk['_comp_key'])
        seen_lyrics_hash.update(chunk['_lyrics_hash'])
        word_counts.extend(chunk['_word_count'].astype(int).tolist())
        rows_kept += len(chunk)
        log('PASS1', f'chunk {i:03d}: kept {len(chunk):6d} | total in {rows_in:7d}')

    if not word_counts:
        raise RuntimeError('No rows survived initial filtering; check the raw CSV and filters.')

    wc_series = pd.Series(word_counts)
    q1, q3 = wc_series.quantile(0.25), wc_series.quantile(0.75)
    iqr = q3 - q1
    lo = max(1, int(q1 - 1.5 * iqr))
    hi = int(q3 + 1.5 * iqr)
    log('PASS1', f'done: input {rows_in:,} -> kept {rows_kept:,}; IQR bounds [{lo}, {hi}]')
    return lo, hi, rows_in, rows_kept


def pass2_write_output(raw_csv: Path, out_csv: Path, chunk_size: int, usecols, lo: int, hi: int) -> int:
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    if out_csv.exists():
        out_csv.unlink()

    seen_comp_keys: set[str] = set()
    seen_lyrics_hash: set[str] = set()
    rows_written = 0

    for i, chunk in enumerate(pd.read_csv(raw_csv, usecols=usecols, chunksize=chunk_size, low_memory=False)):
        chunk = clean_chunk(chunk)
        mask_new = (~chunk['_comp_key'].isin(seen_comp_keys)) & (~chunk['_lyrics_hash'].isin(seen_lyrics_hash))
        chunk = chunk[mask_new]
        seen_comp_keys.update(chunk['_comp_key'])
        seen_lyrics_hash.update(chunk['_lyrics_hash'])
        chunk = chunk[(chunk['_word_count'] >= lo) & (chunk['_word_count'] <= hi)].copy()
        out = chunk[['id', 'title', 'artist', 'normalized_title', 'normalized_artist', 'lyrics', 'normalized_lyrics']]
        mode = 'w' if rows_written == 0 else 'a'
        header = rows_written == 0
        out.to_csv(out_csv, mode=mode, header=header, index=False)
        rows_written += len(out)
        log('PASS2', f'chunk {i:03d}: wrote {len(out):6d} | total out {rows_written:,}')
    log('PASS2', f'complete: final rows {rows_written:,} -> {out_csv}')
    return rows_written

SyntaxError: invalid syntax (4213092911.py, line 16)

In [ ]:
# Run the pipeline (edit parameters here if needed)
def run_pipeline(raw_csv: Path = RAW_CSV, out_csv: Path = OUT_CSV, chunk_size: int = CHUNK_SIZE) -> None:
    if not raw_csv.exists():
        raise FileNotFoundError(f'Raw CSV not found at {raw_csv}')
    chunk_size = max(1_000, int(chunk_size))
    usecols = RAW_COLS
    log('START', f'raw: {raw_csv}')
    log('START', f'out : {out_csv}')
    log('START', f'chunks: {chunk_size}')
    lo, hi, rows_in, rows_kept = pass1_compute_bounds(raw_csv, chunk_size, usecols)
    rows_written = pass2_write_output(raw_csv, out_csv, chunk_size, usecols, lo, hi)
    log('DONE', f'input {rows_in:,} | kept after pass1 {rows_kept:,} | written {rows_written:,}')

# Execute
run_pipeline()